# 2. Matrices as Transformations

A $2 \times 2$ matrix is not just a table of numbers -- it is a **function** that moves every point in the plane to a new location.  
This notebook lets you *see* that function in action.

We set up a coordinate grid decorated with two landmark shapes:
- A **circle** in the bottom-left corner
- A **triangle** in the top-left corner

When you apply a matrix, the grid and both shapes deform together, revealing the geometry of the transformation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, HTML

%matplotlib inline

# ── colour palette ────────────────────────────────────────────────
C_GRID   = "#BDBDBD"
C_CIRCLE = "#2196F3"
C_TRI    = "#F44336"
C_E1     = "#4CAF50"
C_E2     = "#FF9800"

## 2.1 Building the Scene

The helper functions below create:
1. A grid of horizontal and vertical lines (we store each line as a sequence of 2D points so we can transform every point).
2. A **circle** centred at $(-3, -3)$ (bottom-left).
3. A **triangle** centred at $(-3, 3)$ (top-left).
4. The two standard-basis vectors $\mathbf{e}_1, \mathbf{e}_2$.

In [ ]:
def make_grid(extent=4, n_lines=9, pts_per_line=200):
    """
    Return grid lines as (N, 2) arrays.

    Parameters
    ----------
    extent : float
        Grid spans [-extent, extent] in both axes.
    n_lines : int
        Number of lines in each direction.
    pts_per_line : int
        Sampling density per line.

    Returns
    -------
    list of ndarray
        Each element is shape (pts_per_line, 2).
    """
    lines = []
    t = np.linspace(-extent, extent, pts_per_line)
    for v in np.linspace(-extent, extent, n_lines):
        lines.append(np.column_stack([t, np.full_like(t, v)]))          # horizontal
        lines.append(np.column_stack([np.full_like(t, v), t]))          # vertical
    return lines


def make_circle(centre=(-3, -3), radius=0.8, n=120):
    """
    Return points on a filled circle as (n, 2).

    Parameters
    ----------
    centre : tuple
        (x, y) centre of the circle.
    radius : float
        Radius.
    n : int
        Number of boundary samples.

    Returns
    -------
    ndarray, shape (n, 2)
    """
    theta = np.linspace(0, 2 * np.pi, n, endpoint=False)
    pts = np.column_stack([centre[0] + radius * np.cos(theta),
                           centre[1] + radius * np.sin(theta)])
    return pts


def make_triangle(centre=(-3, 3), size=0.9):
    """
    Return vertices of an equilateral triangle as (4, 2) -- closed polygon.

    Parameters
    ----------
    centre : tuple
        (x, y) centroid.
    size : float
        Distance from centroid to each vertex.

    Returns
    -------
    ndarray, shape (4, 2)
    """
    angles = np.array([np.pi / 2, np.pi / 2 + 2 * np.pi / 3, np.pi / 2 + 4 * np.pi / 3])
    verts = np.column_stack([centre[0] + size * np.cos(angles),
                             centre[1] + size * np.sin(angles)])
    return np.vstack([verts, verts[0:1]])  # close the polygon

In [ ]:
GRID   = make_grid()
CIRCLE = make_circle()
TRI    = make_triangle()

## 2.2 Drawing and Transforming

A matrix $M = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$ sends every point $\mathbf{p}$ to $M\mathbf{p}$.  
We simply multiply every point in the grid and each shape by $M$ and re-plot.

Key things to watch:
- The **green arrow** ($\mathbf{e}_1$) maps to the first *column* of $M$.
- The **orange arrow** ($\mathbf{e}_2$) maps to the second *column* of $M$.
- The **determinant** $\det(M) = ad - bc$ tells you the signed area-scaling factor.

In [ ]:
def transform_pts(pts, M):
    """Apply 2x2 matrix M to (N,2) point array."""
    return pts @ M.T


def draw_scene(ax, grid, circle, tri, e1, e2, lim=6, title=""):
    """
    Draw the full scene (grid + shapes + basis vectors) on *ax*.

    Parameters
    ----------
    ax : matplotlib Axes
    grid : list of (N,2) arrays
    circle : (N,2) array
    tri : (4,2) array
    e1, e2 : 1-D arrays, length 2
    lim : float
    title : str
    """
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.axhline(0, color="k", lw=0.4)
    ax.axvline(0, color="k", lw=0.4)

    for line in grid:
        ax.plot(line[:, 0], line[:, 1], color=C_GRID, lw=0.6)

    ax.fill(circle[:, 0], circle[:, 1], color=C_CIRCLE, alpha=0.45)
    ax.plot(circle[:, 0], circle[:, 1], color=C_CIRCLE, lw=1.5)

    ax.fill(tri[:, 0], tri[:, 1], color=C_TRI, alpha=0.45)
    ax.plot(tri[:, 0], tri[:, 1], color=C_TRI, lw=1.5)

    ax.annotate("", xy=e1, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=C_E1, lw=2.5))
    ax.annotate("", xy=e2, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=C_E2, lw=2.5))
    ax.text(e1[0] + 0.15, e1[1] + 0.15, "e1", color=C_E1, fontsize=11, fontweight="bold")
    ax.text(e2[0] + 0.15, e2[1] + 0.15, "e2", color=C_E2, fontsize=11, fontweight="bold")

    if title:
        ax.set_title(title, fontsize=13)

In [ ]:
def show_transformation(M, title_right="Transformed"):
    """
    Side-by-side original / transformed view.

    Parameters
    ----------
    M : ndarray, shape (2, 2)
        Transformation matrix.
    title_right : str
        Title for the right (transformed) plot.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

    draw_scene(ax1, GRID, CIRCLE, TRI,
               e1=np.array([1, 0]), e2=np.array([0, 1]),
               title="Original")

    t_grid   = [transform_pts(l, M) for l in GRID]
    t_circle = transform_pts(CIRCLE, M)
    t_tri    = transform_pts(TRI, M)
    t_e1     = M @ np.array([1, 0])
    t_e2     = M @ np.array([0, 1])

    det = np.linalg.det(M)
    info = f"{title_right}\ndet = {det:.2f}"
    draw_scene(ax2, t_grid, t_circle, t_tri, e1=t_e1, e2=t_e2, title=info)

    plt.tight_layout()
    plt.show()

Let's verify the original scene first:

In [ ]:
show_transformation(np.eye(2), title_right="Identity (no change)")

## 2.3 Interactive Transformation Explorer

Drag the sliders to set the four entries of the $2 \times 2$ matrix and watch the space deform in real time.

In [ ]:
def interactive_transform(a, b, c, d):
    M = np.array([[a, b], [c, d]])
    show_transformation(M, title_right=f"M = [[{a:.1f}, {b:.1f}], [{c:.1f}, {d:.1f}]]")

style = {"description_width": "initial"}
widgets.interact(
    interactive_transform,
    a=widgets.FloatSlider(value=1, min=-3, max=3, step=0.1, description="a (row1, col1)", style=style),
    b=widgets.FloatSlider(value=0, min=-3, max=3, step=0.1, description="b (row1, col2)", style=style),
    c=widgets.FloatSlider(value=0, min=-3, max=3, step=0.1, description="c (row2, col1)", style=style),
    d=widgets.FloatSlider(value=1, min=-3, max=3, step=0.1, description="d (row2, col2)", style=style),
);

## 2.4 Gallery of Named Transformations

Below we apply several classic matrix types so you can build intuition for each.

### 2.4.1 Rotation

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

A rotation preserves all lengths and angles.  The determinant is always 1 (no area change), and the circle stays a circle.

In [ ]:
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
show_transformation(R, title_right=f"Rotation by 45°")

In [ ]:
def interactive_rotation(angle_deg):
    th = np.radians(angle_deg)
    R = np.array([[np.cos(th), -np.sin(th)],
                  [np.sin(th),  np.cos(th)]])
    show_transformation(R, title_right=f"Rotation by {angle_deg:.0f}°")

widgets.interact(
    interactive_rotation,
    angle_deg=widgets.FloatSlider(value=0, min=-180, max=180, step=5,
                                  description="θ (degrees)", style=style),
);

### 2.4.2 Scaling

$$S = \begin{pmatrix} s_x & 0 \\ 0 & s_y \end{pmatrix}$$

Uniform scaling ($s_x = s_y$) keeps the shape the same, just bigger or smaller.  
Non-uniform scaling ($s_x \neq s_y$) stretches differently along each axis -- the circle becomes an ellipse.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

cases = [
    (np.diag([2, 2]),     "Uniform scale (2, 2)"),
    (np.diag([2, 0.5]),   "Non-uniform (2, 0.5)"),
    (np.diag([-1, 1]),    "Reflect x-axis (-1, 1)"),
]

for ax, (M, title) in zip(axes, cases):
    t_grid   = [transform_pts(l, M) for l in GRID]
    t_circle = transform_pts(CIRCLE, M)
    t_tri    = transform_pts(TRI, M)
    t_e1     = M @ np.array([1, 0])
    t_e2     = M @ np.array([0, 1])
    draw_scene(ax, t_grid, t_circle, t_tri, t_e1, t_e2,
               title=f"{title}\ndet = {np.linalg.det(M):.2f}")

plt.tight_layout()
plt.show()

### 2.4.3 Shearing

$$\text{Shear}_x = \begin{pmatrix} 1 & k \\ 0 & 1 \end{pmatrix}, \qquad
\text{Shear}_y = \begin{pmatrix} 1 & 0 \\ k & 1 \end{pmatrix}$$

Shearing slides layers of the plane past each other. The determinant stays 1 (area unchanged).

In [ ]:
def interactive_shear(k, direction):
    if direction == "x-shear":
        M = np.array([[1, k], [0, 1]])
    else:
        M = np.array([[1, 0], [k, 1]])
    show_transformation(M, title_right=f"{direction}, k = {k:.1f}")

widgets.interact(
    interactive_shear,
    k=widgets.FloatSlider(value=0, min=-2, max=2, step=0.1, description="k"),
    direction=widgets.Dropdown(options=["x-shear", "y-shear"], description="Direction"),
);

### 2.4.4 Reflection

Reflection across a line through the origin at angle $\alpha$ is

$$\text{Ref}(\alpha) = \begin{pmatrix} \cos 2\alpha & \sin 2\alpha \\ \sin 2\alpha & -\cos 2\alpha \end{pmatrix}$$

The determinant is $-1$ (orientation is flipped).

In [ ]:
def interactive_reflection(alpha_deg):
    a = np.radians(alpha_deg)
    M = np.array([[np.cos(2*a),  np.sin(2*a)],
                  [np.sin(2*a), -np.cos(2*a)]])
    show_transformation(M, title_right=f"Reflect across line at {alpha_deg:.0f}°")

widgets.interact(
    interactive_reflection,
    alpha_deg=widgets.FloatSlider(value=0, min=0, max=180, step=5,
                                  description="α (degrees)", style=style),
);

### 2.4.5 Projection

Projection onto a line through the origin at angle $\alpha$:

$$P(\alpha) = \begin{pmatrix} \cos^2 \alpha & \cos\alpha \sin\alpha \\ \cos\alpha \sin\alpha & \sin^2 \alpha \end{pmatrix}$$

The determinant is 0 -- the entire plane collapses onto a line (information is lost, the transformation is not invertible).

In [ ]:
def interactive_projection(alpha_deg):
    a = np.radians(alpha_deg)
    M = np.array([[np.cos(a)**2,           np.cos(a)*np.sin(a)],
                  [np.cos(a)*np.sin(a),    np.sin(a)**2]])
    show_transformation(M, title_right=f"Project onto line at {alpha_deg:.0f}°")

widgets.interact(
    interactive_projection,
    alpha_deg=widgets.FloatSlider(value=0, min=0, max=180, step=5,
                                  description="α (degrees)", style=style),
);

## 2.5 Composition of Transformations

Applying matrix $A$ then matrix $B$ is the same as applying the single matrix $BA$ (note the order!).  
Below we rotate by 30° and then shear -- compare with doing it in the opposite order.

In [ ]:
th = np.radians(30)
R = np.array([[np.cos(th), -np.sin(th)],
              [np.sin(th),  np.cos(th)]])
S = np.array([[1, 0.8], [0, 1]])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

draw_scene(axes[0], GRID, CIRCLE, TRI,
           np.array([1, 0]), np.array([0, 1]), title="Original")

M1 = S @ R
t_grid   = [transform_pts(l, M1) for l in GRID]
t_circle = transform_pts(CIRCLE, M1)
t_tri    = transform_pts(TRI, M1)
draw_scene(axes[1], t_grid, t_circle, t_tri,
           M1 @ np.array([1,0]), M1 @ np.array([0,1]),
           title="Shear · Rotation")

M2 = R @ S
t_grid   = [transform_pts(l, M2) for l in GRID]
t_circle = transform_pts(CIRCLE, M2)
t_tri    = transform_pts(TRI, M2)
draw_scene(axes[2], t_grid, t_circle, t_tri,
           M2 @ np.array([1,0]), M2 @ np.array([0,1]),
           title="Rotation · Shear")

plt.tight_layout()
plt.show()

print("Matrix multiplication is NOT commutative -- order matters!")

### 2.5.1 Associativity and Commutativity

Two important properties of matrix multiplication:

- **Associative:** $A \cdot (B \cdot C) = (A \cdot B) \cdot C$ — this **always** holds.  
  This is why you can pre-multiply transformations into a single matrix before applying it to a vector.

- **NOT Commutative:** $A \cdot B \neq B \cdot A$ in general — **order matters!**  
  Rotating then shearing is different from shearing then rotating.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[0, 1],
              [1, 0]])

C = np.array([[2, 0],
              [0, 3]])

# --- Associativity: A(BC) should equal (AB)C ---
LHS_assoc = A @ (B @ C)
RHS_assoc = (A @ B) @ C

print("=== Associativity ===")
print(f"A =\n{A}\n")
print(f"B =\n{B}\n")
print(f"C =\n{C}\n")
print(f"A @ (B @ C) =\n{LHS_assoc}\n")
print(f"(A @ B) @ C =\n{RHS_assoc}\n")
print(f"Equal? {np.allclose(LHS_assoc, RHS_assoc)}\n")

# --- Commutativity: AB vs BA ---
AB = A @ B
BA = B @ A

print("=== Commutativity ===")
print(f"A @ B =\n{AB}\n")
print(f"B @ A =\n{BA}\n")
print(f"Equal? {np.allclose(AB, BA)}")
print("\nMatrix multiplication is associative but NOT commutative!")

### 2.5.2 Matrix Transpose

The **transpose** $M^T$ swaps rows and columns:

$$\begin{pmatrix} a & b \\ c & d \end{pmatrix}^T = \begin{pmatrix} a & c \\ b & d \end{pmatrix}$$

**Key intuitions:**

| Intuition | Explanation |
|-----------|-------------|
| Mechanical | Row $i$ of $M$ becomes column $i$ of $M^T$ (mirror across the main diagonal) |
| Dot product | $\mathbf{v} \cdot \mathbf{w} = \mathbf{v}^T \mathbf{w}$ — transpose turns a column vector into a row vector for the dot product |
| Geometric | For transformations: $\langle M\mathbf{v},\; \mathbf{w}\rangle = \langle \mathbf{v},\; M^T\mathbf{w}\rangle$ — the transpose "moves $M$ to the other side" of an inner product |
| Rotation special case | For rotation matrices, $R^T = R^{-1}$ — the transpose *undoes* the rotation |
| Non-square matrices | An $m \times n$ matrix becomes an $n \times m$ matrix — the transpose swaps the domain and codomain |

**Useful identities:**

- $(M^T)^T = M$
- $(AB)^T = B^T A^T$ (note the reversed order!)
- $(M + N)^T = M^T + N^T$

In [ ]:
M = np.array([[1, 2],
              [3, 4]])

print("=== Basic Transpose ===")
print(f"M =\n{M}\n")
print(f"M^T =\n{M.T}\n")

# --- Dot product via transpose ---
v = np.array([[3], [1]])
w = np.array([[2], [5]])

dot_direct = np.dot(v.flatten(), w.flatten())
dot_transpose = (v.T @ w)[0, 0]

print("=== Dot Product via Transpose ===")
print(f"v = {v.flatten()},  w = {w.flatten()}")
print(f"v . w (np.dot)     = {dot_direct}")
print(f"v^T @ w            = {dot_transpose}\n")

# --- (AB)^T = B^T A^T ---
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("=== (AB)^T = B^T A^T ===")
print(f"(A @ B)^T =\n{(A @ B).T}\n")
print(f"B^T @ A^T =\n{B.T @ A.T}\n")
print(f"Equal? {np.allclose((A @ B).T, B.T @ A.T)}\n")

# --- Rotation matrix: R^T = R^{-1} ---
theta = np.radians(45)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])

print("=== Rotation: R^T = R^{-1} ===")
print(f"R (45° rotation) =\n{np.round(R, 4)}\n")
print(f"R^T =\n{np.round(R.T, 4)}\n")
print(f"R @ R^T (should be identity) =\n{np.round(R @ R.T, 10)}")
print(f"Is R^T == R^(-1)? {np.allclose(R.T, np.linalg.inv(R))}")

## 2.6 The Determinant as Area Factor

| det(M) | Meaning |
|--------|---------|
| > 0    | Areas scale by det(M), orientation preserved |
| < 0    | Areas scale by |det(M)|, orientation **flipped** |
| = 0    | Entire plane collapses to a line or point (singular matrix) |

Try the interactive explorer above: as you push the determinant toward zero, the shapes flatten.

## 2.7 Matrix Columns = Where the Basis Goes

This is perhaps the single most important insight:

> The columns of a matrix tell you where the standard basis vectors land.

If $M = [\mathbf{m}_1 \;|\; \mathbf{m}_2]$ then $M \mathbf{e}_1 = \mathbf{m}_1$ and $M \mathbf{e}_2 = \mathbf{m}_2$.  
Every other vector is a linear combination of $\mathbf{e}_1$ and $\mathbf{e}_2$, so it transforms accordingly.

Watch the green and orange arrows in the visualisations above -- they always land at the matrix columns.

In [ ]:
M_example = np.array([[2, 1], [0.5, 1.5]])
print(f"Matrix M:\n{M_example}")
print(f"\nColumn 1 (where e1 goes): {M_example[:, 0]}")
print(f"Column 2 (where e2 goes): {M_example[:, 1]}")

show_transformation(M_example, title_right="Columns = basis destinations")

---

## Exercises

### Exercise 1

Without running any code, predict what the matrix $\begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ does to the plane.  
Hint: where does $\mathbf{e}_1$ go? Where does $\mathbf{e}_2$ go?

<details>
<summary><b>Solution</b></summary>

$\mathbf{e}_1 = (1,0)^T \mapsto (0,1)^T$ and $\mathbf{e}_2 = (0,1)^T \mapsto (-1,0)^T$.  
This is a **90° counter-clockwise rotation**.

</details>

In [ ]:
# Verify Exercise 1
show_transformation(np.array([[0, -1], [1, 0]]), title_right="Exercise 1 check")

### Exercise 2

Construct a $2 \times 2$ matrix that first rotates by 60° and then scales the $x$-component by 2.  
Compute the matrix and verify with `show_transformation`.

<details>
<summary><b>Solution</b></summary>

Rotation by 60°: $R = \begin{pmatrix} \cos60° & -\sin60° \\ \sin60° & \cos60° \end{pmatrix}$  
Scaling: $S = \begin{pmatrix} 2 & 0 \\ 0 & 1 \end{pmatrix}$  
Combined (scale **after** rotation): $M = S R$

</details>

In [ ]:
# Verify Exercise 2
th = np.radians(60)
R = np.array([[np.cos(th), -np.sin(th)],
              [np.sin(th),  np.cos(th)]])
S = np.diag([2, 1])
M = S @ R
print(f"Combined matrix:\n{M}")
show_transformation(M, title_right="Scale-x(2) after Rotate(60°)")

### Exercise 3

Use the interactive slider to find a matrix whose determinant is close to zero.  
Describe what happens to the grid and shapes. Why is this matrix not invertible?

<details>
<summary><b>Solution</b></summary>

When $\det(M) \approx 0$ the grid and shapes collapse toward a line (or point if $M = 0$).  
The matrix is not invertible because the transformation loses information: multiple input points map to the same output.

</details>

In [ ]:
M = np.array([[1e-2000, 0],[0, 1]])
print(f"{M=}")
np.linalg.det(M)
np.linalg.inv(M)
print(np.isclose(np.linalg.det(M), 0))
# np.linalg.inv(M)